In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_ROOT = Path().resolve().parent
DB_PATH = PROJECT_ROOT / "data" / "cineanalytica.db"
conn = sqlite3.connect(DB_PATH)

movies = pd.read_sql_query("""
SELECT movie_id, title, overview
FROM movies
""", conn)

# Optional: if you have a movie_genres table, we can join it next
movies.head(), movies.shape

(   movie_id            title  \
 0         5       Four Rooms   
 1        11        Star Wars   
 2        12     Finding Nemo   
 3        13     Forrest Gump   
 4        14  American Beauty   
 
                                             overview  
 0  It's Ted the Bellhop's first night on the job....  
 1  Princess Leia is captured and held hostage by ...  
 2  Nemo, an adventurous young clownfish, is unexp...  
 3  A man with a low IQ has accomplished great thi...  
 4  Lester Burnham, a depressed suburban father in...  ,
 (4803, 3))

In [2]:
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", conn)

,name
0,actors
1,directors
2,genres
3,movie_cast
4,movie_directors
5,movie_features
6,movie_genres
7,movies
8,ratings
9,reviews


In [3]:
genres_df = pd.read_sql_query("""
SELECT mg.movie_id, g.genre_name
FROM movie_genres mg
JOIN genres g ON mg.genre_id = g.genre_id
""", conn)

# Combine genres into one string per movie
genre_text = genres_df.groupby("movie_id")["genre_name"].apply(lambda x: " ".join(sorted(set(x)))).reset_index()
genre_text = genre_text.rename(columns={"genre_name": "genre_text"})

movies2 = movies.merge(genre_text, on="movie_id", how="left")
movies2["overview"] = movies2["overview"].fillna("")
movies2["genre_text"] = movies2["genre_text"].fillna("")

movies2.head(), movies2.shape

(   movie_id            title  \
 0         5       Four Rooms   
 1        11        Star Wars   
 2        12     Finding Nemo   
 3        13     Forrest Gump   
 4        14  American Beauty   
 
                                             overview  \
 0  It's Ted the Bellhop's first night on the job....   
 1  Princess Leia is captured and held hostage by ...   
 2  Nemo, an adventurous young clownfish, is unexp...   
 3  A man with a low IQ has accomplished great thi...   
 4  Lester Burnham, a depressed suburban father in...   
 
                          genre_text  
 0                      Comedy Crime  
 1  Action Adventure Science Fiction  
 2                  Animation Family  
 3              Comedy Drama Romance  
 4                             Drama  ,
 (4803, 4))

In [4]:
movies2["combined_text"] = (movies2["overview"] + " " + movies2["genre_text"]).str.strip()
movies2[["title", "genre_text"]].head()

,title,genre_text
0,Four Rooms,Comedy Crime
1,Star Wars,Action Adventure Science Fiction
2,Finding Nemo,Animation Family
3,Forrest Gump,Comedy Drama Romance
4,American Beauty,Drama


In [6]:
## TF-IDF vectorization + cosine similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

tfidf = TfidfVectorizer(stop_words="english", max_features=50000, ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(movies2["combined_text"])

cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

# mapping title -> index
title_to_idx = pd.Series(movies2.index, index=movies2["title"]).drop_duplicates()

tfidf_matrix.shape

(4803, 50000)

In [7]:
## Recommendation Function
def recommend(title, top_n=10):
    if title not in title_to_idx:
        # quick fuzzy suggestion
        matches = movies2[movies2["title"].str.contains(title, case=False, na=False)]["title"].head(10).tolist()
        return {"error": f"Title not found: {title}", "did_you_mean": matches}

    idx = title_to_idx[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # exclude itself
    sim_scores = sim_scores[1:top_n+1]
    movie_indices = [i for i, score in sim_scores]

    out = movies2.loc[movie_indices, ["movie_id", "title", "genre_text"]].copy()
    out["similarity"] = [score for i, score in sim_scores]
    return out.reset_index(drop=True)

In [10]:
#test 
recommend("The Dark Knight", top_n=10)

,movie_id,title,genre_text,similarity
0,49026,The Dark Knight Rises,Action Crime Drama Thriller,0.282655
1,414,Batman Forever,Action Crime Fantasy,0.214632
2,364,Batman Returns,Action Fantasy,0.177600
3,268,Batman,Action Fantasy,0.154265
4,18777,Slow Burn,Crime Drama Mystery Thriller,0.136545
5,142061,"Batman: The Dark Knight Returns, Part 2",Action Animation,0.129852
6,22803,Law Abiding Citizen,Crime Drama Thriller,0.121880
7,272,Batman Begins,Action Crime Drama,0.114471
8,820,JFK,Drama History Thriller,0.112614
9,58574,Sherlock Holmes: A Game of Shadows,Action Adventure Crime Mystery,0.093718


In [11]:
ratings = pd.read_sql_query("""
SELECT user_id, movie_id, rating
FROM ratings
""", conn)

ratings.head(), ratings.shape

(   user_id  movie_id  rating
 0        1       862     4.0
 1        1       807     5.0
 2        1       629     5.0
 3        1       755     3.0
 4        1     13685     5.0,
 (70193, 3))

In [12]:
ratings["user_id"].nunique(), ratings["movie_id"].nunique(), ratings["rating"].describe()

(610,
 3535,
 count    70193.000000
 mean         3.535930
 std          1.035333
 min          0.500000
 25%          3.000000
 50%          3.500000
 75%          4.000000
 max          5.000000
 Name: rating, dtype: float64)

In [13]:
# Collaborative Filtering (SVD)
import surprise

In [14]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise.accuracy import rmse

# Surprise expects the rating scale
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[["user_id", "movie_id", "rating"]], reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

svd = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd.fit(trainset)

preds = svd.test(testset)
rmse_val = rmse(preds, verbose=True)

rmse_val

RMSE: 0.8675


0.8674522407743334

In [15]:
import numpy as np
import pandas as pd

all_movie_ids = movies2["movie_id"].unique()

def recommend_for_user(user_id, top_n=10):
    # movies already rated by this user
    rated = ratings.loc[ratings["user_id"] == user_id, "movie_id"].unique()
    candidates = np.setdiff1d(all_movie_ids, rated)

    # predict ratings for candidate movies
    preds = [(mid, svd.predict(user_id, mid).est) for mid in candidates]
    preds.sort(key=lambda x: x[1], reverse=True)

    top = preds[:top_n]
    out = pd.DataFrame(top, columns=["movie_id", "pred_rating"]).merge(
        movies2[["movie_id", "title", "genre_text"]],
        on="movie_id",
        how="left"
    )
    return out

# test with a real user
test_user = int(ratings["user_id"].iloc[0])
recommend_for_user(test_user, top_n=10)

,movie_id,pred_rating,title,genre_text
0,120,5.000000,The Lord of the Rings: The Fellowship of the Ring,Action Adventure Fantasy
1,122,5.000000,The Lord of the Rings: The Return of the King,Action Adventure Fantasy
2,155,5.000000,The Dark Knight,Action Crime Drama Thriller
3,935,5.000000,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy Drama War
4,8321,5.000000,In Bruges,Comedy Crime Drama
5,9388,5.000000,Thank You for Smoking,Comedy Drama
6,239,4.963446,Some Like It Hot,Comedy Romance
7,11013,4.952461,Secretary,Comedy Drama Romance
8,510,4.946235,One Flew Over the Cuckoo's Nest,Drama
9,121,4.945636,The Lord of the Rings: The Two Towers,Action Adventure Fantasy


In [16]:
import joblib
from pathlib import Path

MODEL_DIR = Path().resolve().parent / "models"
MODEL_DIR.mkdir(exist_ok=True)

reco_artifact = {
    "svd": svd,
    "tfidf": tfidf,
    "cosine_sim": cosine_sim,
    "title_to_idx": title_to_idx.to_dict(),
    "movies_lookup": movies2[["movie_id", "title", "genre_text"]].copy()
}

joblib.dump(reco_artifact, MODEL_DIR / "recommender_artifacts.joblib")

print("Saved:", MODEL_DIR / "recommender_artifacts.joblib")

Saved: /Users/khushdomadiya/Desktop/CineAnalytica/models/recommender_artifacts.joblib


In [17]:
## strealit function integration
import numpy as np
import pandas as pd

def recommend_by_title(title, artifact, top_n=10):
    movies_lookup = artifact["movies_lookup"]
    title_to_idx = artifact["title_to_idx"]
    cosine_sim = artifact["cosine_sim"]

    if title not in title_to_idx:
        matches = movies_lookup[movies_lookup["title"].str.contains(title, case=False, na=False)]["title"].head(10).tolist()
        return {"error": f"Title not found: {title}", "did_you_mean": matches}

    idx = title_to_idx[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    movie_indices = [i for i, _ in sim_scores]
    out = movies_lookup.iloc[movie_indices].copy()
    out["similarity"] = [s for _, s in sim_scores]
    return out.reset_index(drop=True)

def recommend_for_user(user_id, artifact, ratings_df, top_n=10):
    svd = artifact["svd"]
    movies_lookup = artifact["movies_lookup"]
    all_movie_ids = movies_lookup["movie_id"].unique()

    rated = ratings_df.loc[ratings_df["user_id"] == user_id, "movie_id"].unique()
    candidates = np.setdiff1d(all_movie_ids, rated)

    preds = [(mid, svd.predict(user_id, mid).est) for mid in candidates]
    preds.sort(key=lambda x: x[1], reverse=True)
    top = preds[:top_n]

    out = pd.DataFrame(top, columns=["movie_id", "pred_rating"]).merge(
        movies_lookup, on="movie_id", how="left"
    )
    return out

In [20]:
loaded = joblib.load(MODEL_DIR / "recommender_artifacts.joblib")
recommend_by_title("The Dark Knight", loaded, top_n=10)

,movie_id,title,genre_text,similarity
0,49026,The Dark Knight Rises,Action Crime Drama Thriller,0.282655
1,414,Batman Forever,Action Crime Fantasy,0.214632
2,364,Batman Returns,Action Fantasy,0.177600
3,268,Batman,Action Fantasy,0.154265
4,18777,Slow Burn,Crime Drama Mystery Thriller,0.136545
5,142061,"Batman: The Dark Knight Returns, Part 2",Action Animation,0.129852
6,22803,Law Abiding Citizen,Crime Drama Thriller,0.121880
7,272,Batman Begins,Action Crime Drama,0.114471
8,820,JFK,Drama History Thriller,0.112614
9,58574,Sherlock Holmes: A Game of Shadows,Action Adventure Crime Mystery,0.093718


In [19]:
recommend_for_user(test_user, loaded, ratings, top_n=10).head()

,movie_id,pred_rating,title,genre_text
0,120,5.0,The Lord of the Rings: The Fellowship of the Ring,Action Adventure Fantasy
1,122,5.0,The Lord of the Rings: The Return of the King,Action Adventure Fantasy
2,155,5.0,The Dark Knight,Action Crime Drama Thriller
3,935,5.0,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy Drama War
4,8321,5.0,In Bruges,Comedy Crime Drama


In [21]:
import joblib
from pathlib import Path

MODEL_DIR = Path().resolve().parent / "models"
MODEL_DIR.mkdir(exist_ok=True)

svd_artifact = {
    "svd": svd,
    "movies_lookup": movies2[["movie_id", "title", "genre_text"]].copy()
}

joblib.dump(svd_artifact, MODEL_DIR / "svd_recommender.joblib")
print("Saved:", MODEL_DIR / "svd_recommender.joblib")

Saved: /Users/khushdomadiya/Desktop/CineAnalytica/models/svd_recommender.joblib


In [22]:
ls -lh models/svd_recommender.joblib

ls: models/svd_recommender.joblib: No such file or directory
